# Header & Precompiled Header (PCH) Analysis

Analyse header inclusion patterns, identify PCH candidates, and simulate PCH impact on compile times.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
CONFIG_DIR = Path("../../data/config")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
from buildanalysis.headers import identify_pch_candidates, simulate_pch_impact, analyse_pch_opportunities, analyse_pch_overlap
from buildanalysis.graph import build_include_graph

## 1. Header Inclusion Overview

Summary of header inclusion patterns, system vs project headers, and include depth distribution.

In [ ]:
if ds.has_file("header_edges"):
    header_edges = ds.header_edges
    
    # Use the is_system column from header_edges (computed by build_header_edges.py
    # using path prefix matching, NOT angle-bracket detection)
    system_edges = header_edges[header_edges["is_system"]]
    project_edges = header_edges[~header_edges["is_system"]]
    
    system_headers = header_edges[header_edges["is_system"]]["included"].unique()
    project_headers = header_edges[~header_edges["is_system"]]["included"].unique()
    all_unique = pd.concat([header_edges["includer"], header_edges["included"]]).unique()
    
    print(f"Header Inclusion Overview:")
    print(f"  Total unique headers: {len(all_unique)}")
    print(f"  System headers: {len(system_headers)} ({len(system_headers)/len(all_unique)*100:.1f}%)")
    print(f"  Project headers: {len(project_headers)} ({len(project_headers)/len(all_unique)*100:.1f}%)")
    print(f"  Total include edges: {len(header_edges)}")
    print(f"  System include edges: {len(system_edges)} ({len(system_edges)/len(header_edges)*100:.1f}%)")
    print(f"  Project include edges: {len(project_edges)} ({len(project_edges)/len(header_edges)*100:.1f}%)")
    print(f"  Mean include depth: {header_edges['depth'].mean():.2f}")
else:
    print("Header edges data not available. Skipping header analysis.")

In [ ]:
if ds.has_file("header_edges"):
    depth_distribution = header_edges['depth'].value_counts().sort_index()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(depth_distribution.index, depth_distribution.values, color='steelblue', alpha=0.7)
    ax.set_xlabel('Include Depth', fontsize=11)
    ax.set_ylabel('Number of Edges', fontsize=11)
    ax.set_title('Distribution of Include Depths', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

## 2. Most Included Headers

Top 30 headers by inclusion count, distinguishing system vs project headers.

In [ ]:
if ds.has_file("header_edges"):
    # Build a lookup of header path -> is_system from the edges data
    header_system_map = header_edges.drop_duplicates("included").set_index("included")["is_system"].to_dict()
    
    inclusion_counts = header_edges["included"].value_counts()
    top_included = inclusion_counts.head(30)
    
    print(f"Top 30 Most Included Headers:")
    print()
    for i, (header, count) in enumerate(top_included.items(), 1):
        is_sys = header_system_map.get(header, False)
        header_type = "[SYSTEM]" if is_sys else "[PROJECT]"
        # Shorten display for readability
        display_name = header if len(header) < 70 else "..." + header[-67:]
        print(f"  {i:2d}. {header_type:10s} {display_name:70s} {count:4d} times")

In [ ]:
if ds.has_file("header_edges"):
    fig, ax = plt.subplots(figsize=(12, 8))
    
    top_20 = inclusion_counts.head(20)
    colors = ["coral" if header_system_map.get(h, False) else "steelblue" for h in top_20.index]
    
    short_labels = [h if len(h) < 50 else "..." + h[-47:] for h in top_20.index]
    
    ax.barh(range(len(top_20)), top_20.values, color=colors)
    ax.set_yticks(range(len(top_20)))
    ax.set_yticklabels(short_labels, fontsize=9)
    ax.set_xlabel("Inclusion Count", fontsize=11)
    ax.set_title("Top 20 Most Included Headers", fontsize=12, fontweight="bold")
    ax.invert_yaxis()
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor="steelblue", label="Project"), Patch(facecolor="coral", label="System")]
    ax.legend(handles=legend_elements, loc="lower right")
    
    plt.tight_layout()
    plt.show()

## 3. Include Graph Analysis

Build and analyse the include dependency graph.

In [ ]:
if ds.has_file("header_edges"):
    include_graph = build_include_graph(header_edges)
    
    print(f"Include Graph Summary:")
    print(f"  Headers (nodes): {include_graph.number_of_nodes()}")
    print(f"  Include edges: {include_graph.number_of_edges()}")
    print(f"  Density: {include_graph.number_of_edges() / (include_graph.number_of_nodes() * (include_graph.number_of_nodes() - 1)):.4f}")
    
    in_degrees = {n: include_graph.in_degree(n) for n in include_graph.nodes()}
    out_degrees = {n: include_graph.out_degree(n) for n in include_graph.nodes()}
    
    print(f"\nTop 10 Headers by In-Degree (most included):")
    for header, degree in sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {header:60s} {degree:4d} incoming")
    
    print(f"\nTop 10 Headers by Out-Degree (most includes):")
    for header, degree in sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {header:60s} {degree:4d} outgoing")

## 4. PCH Candidate Analysis

Identify precompiled header candidates per target.

In [ ]:
if ds.has_file("header_edges"):
    target_metrics = ds.target_metrics
    file_metrics = ds.file_metrics
    header_metrics_df = ds.header_metrics if ds.has_file("header_metrics") else None
    
    multi_file_targets = target_metrics[target_metrics["file_count"] >= 3]["cmake_target"].tolist()
    
    print(f"PCH Candidate Analysis:")
    print(f"  Targets with >=3 source files: {len(multi_file_targets)}")
    print()
    
    # Build git churn data for stability scoring
    git_churn_df = file_metrics[["source_file", "git_commit_count", "git_churn"]].rename(
        columns={"git_commit_count": "n_commits", "git_churn": "total_churn"}
    ) if "git_commit_count" in file_metrics.columns else pd.DataFrame(columns=["source_file", "n_commits", "total_churn"])
    
    if header_metrics_df is not None:
        # Use the library's batch analysis function with correct signature
        pch_results = analyse_pch_opportunities(
            targets=multi_file_targets,
            include_graph=include_graph,
            file_metrics=file_metrics,
            header_metrics=header_metrics_df,
            git_churn=git_churn_df,
            n_candidates_per_target=15,
        )
        
        print(f"  Targets with PCH candidates: {len(pch_results)}")
        
        if len(pch_results) > 0:
            # Display per-target results
            display_cols = [c for c in ["cmake_target", "source_file_count", "pch_header_count",
                                         "estimated_savings_ms", "savings_fraction", "recommendation"]
                           if c in pch_results.columns]
            print(f"\nPCH Analysis Results (sorted by estimated savings):")
            print(pch_results[display_cols].to_string(index=False))
            
            # Collect individual PCH candidates for each target
            all_pch_candidates = []
            for _, result_row in pch_results.iterrows():
                target_name = result_row["cmake_target"]
                candidates = identify_pch_candidates(
                    target=target_name,
                    include_graph=include_graph,
                    file_metrics=file_metrics,
                    header_metrics=header_metrics_df,
                    git_churn=git_churn_df,
                    n_candidates=10,
                )
                if len(candidates) > 0:
                    for _, cand in candidates.head(3).iterrows():
                        all_pch_candidates.append({
                            "target": target_name,
                            "header": cand["header_file"],
                            "score": cand["pch_score"],
                            "coverage": cand.get("coverage", 0),
                        })
            
            if all_pch_candidates:
                pch_candidates_df = pd.DataFrame(all_pch_candidates)
                top_global_pch = pch_candidates_df.drop_duplicates("header").nlargest(20, "score")
                
                print(f"\nTop 20 PCH Candidates Across All Targets:")
                for _, row in top_global_pch.iterrows():
                    print(f"  {row['header']:60s} score={row['score']:.3f} (in {row['target']})")
            else:
                pch_candidates_df = pd.DataFrame(columns=["target", "header", "score"])
                top_global_pch = pch_candidates_df
                print("\nNo individual PCH candidates found.")
        else:
            pch_candidates_df = pd.DataFrame(columns=["target", "header", "score"])
            top_global_pch = pch_candidates_df
            pch_results = pd.DataFrame()
            print("\nNo targets produced PCH candidates.")
    else:
        print("WARNING: header_metrics.parquet not available — cannot run PCH analysis")
        pch_candidates_df = pd.DataFrame(columns=["target", "header", "score"])
        top_global_pch = pch_candidates_df
        pch_results = pd.DataFrame()

## 5. PCH Impact Simulation

Estimate compile time savings from implementing PCH for top candidates.

In [ ]:
if ds.has_file("header_edges") and 'pch_results' in locals() and len(pch_results) > 0:
    print("PCH Impact Simulation per Target:")
    print()
    
    pch_impacts = []
    for _, result_row in pch_results.iterrows():
        target_name = result_row["cmake_target"]
        
        # Get per-target PCH candidate headers
        candidates = identify_pch_candidates(
            target=target_name,
            include_graph=include_graph,
            file_metrics=file_metrics,
            header_metrics=header_metrics_df,
            git_churn=git_churn_df,
            n_candidates=15,
        )
        
        if len(candidates) == 0:
            continue
        
        pch_headers = candidates["header_file"].tolist()
        
        impact = simulate_pch_impact(
            target=target_name,
            pch_headers=pch_headers,
            include_graph=include_graph,
            file_metrics=file_metrics,
            header_metrics=header_metrics_df,
            git_churn=git_churn_df,
        )
        
        savings_ms = impact.get("estimated_compile_time_saved_ms", 0)
        recommendation = impact.get("recommendation", "unknown")
        pch_count = impact.get("pch_header_count", 0)
        
        pch_impacts.append({
            "target": target_name,
            "pch_headers": pch_count,
            "savings_ms": savings_ms,
            "recommendation": recommendation,
        })
        print(f"  {target_name:40s} → {savings_ms:8,.0f} ms saved ({pch_count} PCH headers, {recommendation})")
    
    if pch_impacts:
        impacts_df = pd.DataFrame(pch_impacts)
        total_potential_savings = impacts_df["savings_ms"].sum()
        print(f"\n  Total Potential Savings: {total_potential_savings:,.0f} ms")
        
        recommended = impacts_df[impacts_df["recommendation"] == "recommended"]
        print(f"  Targets recommended for PCH: {len(recommended)}")
    else:
        impacts_df = pd.DataFrame(columns=["target", "pch_headers", "savings_ms", "recommendation"])
        print("\nNo PCH impact data to report.")
else:
    impacts_df = pd.DataFrame(columns=["target", "pch_headers", "savings_ms", "recommendation"])
    print("Skipping PCH impact simulation (no PCH candidates found).")

In [ ]:
if ds.has_file("header_edges") and "impacts_df" in locals() and len(impacts_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    sorted_impacts = impacts_df.nlargest(15, "savings_ms")
    y_pos = np.arange(len(sorted_impacts))
    
    colors = ["seagreen" if r == "recommended" else "goldenrod" if r == "marginal" else "lightcoral"
              for r in sorted_impacts["recommendation"].values]
    
    ax.barh(y_pos, sorted_impacts["savings_ms"].values, color=colors, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_impacts["target"].values, fontsize=9)
    ax.set_xlabel("Estimated Compile Time Savings (ms)", fontsize=11)
    ax.set_title("PCH Impact: Compile Time Savings Potential", fontsize=12, fontweight="bold")
    ax.invert_yaxis()
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="seagreen", label="Recommended"),
        Patch(facecolor="goldenrod", label="Marginal"),
        Patch(facecolor="lightcoral", label="Not recommended"),
    ]
    ax.legend(handles=legend_elements, loc="lower right")
    
    plt.tight_layout()
    plt.show()

## 6. Cross-Target PCH Overlap

Identify headers that are PCH candidates across multiple targets — highest-value investments.

In [ ]:
if ds.has_file("header_edges") and "all_pch_candidates" in locals() and len(all_pch_candidates) > 0:
    pch_candidates_df_all = pd.DataFrame(all_pch_candidates)
    
    header_overlap = pch_candidates_df_all.groupby("header").agg({
        "target": "count",
        "score": "mean"
    }).rename(columns={"target": "num_targets"}).sort_values("num_targets", ascending=False)
    
    print(f"Cross-Target PCH Overlap:")
    print(f"  Total unique PCH candidate headers: {len(header_overlap)}")
    print(f"  Headers in multiple targets: {(header_overlap['num_targets'] > 1).sum()}")
    print()
    print(f"Top 20 Headers by Cross-Target Applicability:")
    for header, row in header_overlap.head(20).iterrows():
        short_name = header if len(header) < 55 else "..." + header[-52:]
        print(f"  {short_name:55s} applicable to {row['num_targets']:3.0f} targets (mean score={row['score']:.3f})")
else:
    header_overlap = None
    print("No cross-target PCH overlap data available.")

## 7. Save Intermediates

Persist PCH analysis results for downstream consumption.

In [ ]:
if ds.has_file("header_edges"):
    from buildanalysis.headers import compute_include_fan_metrics, compute_header_impact_score, compute_header_pagerank, compute_include_amplification
    
    # Compute and save header impact metrics (needed by NB08 for include_graph.gexf)
    fan_metrics = compute_include_fan_metrics(include_graph)
    
    header_metrics_df = ds.header_metrics if ds.has_file("header_metrics") else None
    
    # Git churn for headers
    git_churn_df = file_metrics[["source_file", "git_commit_count", "git_churn"]].rename(
        columns={"git_commit_count": "n_commits", "git_churn": "total_churn"}
    ) if "git_commit_count" in file_metrics.columns else pd.DataFrame(columns=["source_file", "n_commits", "total_churn"])
    
    if header_metrics_df is not None:
        header_impact = compute_header_impact_score(fan_metrics, header_metrics_df, git_churn_df)
    else:
        header_impact = fan_metrics.copy()
        header_impact["impact_score"] = 0.0
    
    ds.save_intermediate("header_impact", header_impact)
    print(f"Saved header_impact.parquet ({len(header_impact)} rows)")
    
    # Compute and save amplification ratios
    amplification = compute_include_amplification(include_graph, file_metrics)
    ds.save_intermediate("amplification", amplification)
    print(f"Saved amplification.parquet ({len(amplification)} rows)")
    
    # PCH candidates
    if 'pch_candidates_df' in locals() and not pch_candidates_df.empty:
        ds.save_intermediate("pch_candidates", pch_candidates_df)
        print(f"Saved pch_candidates.parquet ({len(pch_candidates_df)} rows)")
    
    if 'header_overlap' in locals():
        ds.save_intermediate("pch_overlap", header_overlap.reset_index())
        print(f"Saved pch_overlap.parquet")
else:
    print("No header data to save.")